# 05 — Team Stats Season: Full Review

Deep dive into `team_stats_season.parquet` — the raw reference file with **all teams** across all competitions and seasons.  
Goal: understand what we have, what we're missing, and what we can build for the model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATA_ROOT = '/Users/jorgepadilla/Documents/Documents - Jorge\u2019s MacBook Air/thesis_data'
ts = pd.read_parquet(f'{DATA_ROOT}/raw_data/Teams_stats/team_stats_season.parquet')
master = pd.read_parquet(f'{DATA_ROOT}/processed_data/master_dataset/transfers_model_2018_2025.parquet')

print(f'team_stats_season: {ts.shape[0]:,} rows x {ts.shape[1]} cols')
print(f'master parquet:    {master.shape[0]:,} rows x {master.shape[1]} cols')

---
## 1. Basic Structure

In [ ]:
# ID columns vs metric columns
id_cols = ['team_id', 'competition_id', 'season']
metric_cols = [c for c in ts.columns if c not in id_cols]

print(f'ID columns:     {id_cols}')
print(f'Metric columns: {len(metric_cols)}')
print(f'\nDtypes:')
print(ts.dtypes.value_counts().to_string())
print(f'\nNull summary: {ts.isna().sum().sum()} total nulls across entire dataframe')
print(f'Columns with any nulls: {(ts.isna().sum() > 0).sum()}')

# Show columns with nulls
null_cols = ts.isna().sum()
null_cols = null_cols[null_cols > 0]
if len(null_cols) > 0:
    print(f'\nNull detail:')
    for c, n in null_cols.items():
        print(f'  {c}: {n} nulls ({n/len(ts):.1%})')

---
## 2. Coverage: Seasons, Competitions, Teams

In [ ]:
print('=== TEMPORAL COVERAGE ===')
season_counts = ts.groupby('season').agg(
    teams=('team_id', 'nunique'),
    competitions=('competition_id', 'nunique'),
    rows=('team_id', 'count')
).sort_index()
print(season_counts.to_string())

print(f'\n=== TOTALS ===')
print(f'Seasons:      {ts["season"].nunique()} ({ts["season"].min()} to {ts["season"].max()})')
print(f'Competitions: {ts["competition_id"].nunique()}')
print(f'Teams:        {ts["team_id"].nunique()}')
print(f'Unique (team, comp, season): {ts.groupby(["team_id","competition_id","season"]).ngroups}')

In [ ]:
# Visual: rows per season
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

season_counts['rows'].plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Rows per Season')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

season_counts['competitions'].plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Competitions per Season')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# How many teams per competition (distribution)?
teams_per_comp = ts.groupby(['competition_id', 'season'])['team_id'].nunique()
print('Teams per (competition, season):')
print(teams_per_comp.describe().to_string())
print(f'\nSmall competitions (<6 teams): {(teams_per_comp < 6).sum()}')
print(f'Large competitions (>24 teams): {(teams_per_comp > 24).sum()}')

---
## 3. Column Taxonomy

Group the 74 metrics into logical domains.

In [ ]:
# Manual grouping of the 74 metric columns into domains
domains = {
    'Shooting & Goals': [
        'shots', 'shots_on_target', 'goals', 'np_goals', 'own_goals',
        'np_shots', 'high_opportunity_shots', 'penalties',
        'shots_from_outside_box_pct', 'np_xg_per_shot',
        'num_shots_from_sustained_attacks', 'num_shots_from_direct_attacks',
        'shots_from_sustained_attacks_pct', 'shots_from_direct_attacks_pct',
        'shots_per_final_third_pass',
    ],
    'Expected Metrics (xG/xT)': [
        'xg', 'np_xg', 'xt',
        'xg_from_sustained_attacks', 'xg_from_direct_attacks',
        'xg_within_10s_after_recovery', 'xt_within_10s_after_recovery',
    ],
    'Possession & Build-up': [
        'ball_possession_pct', 'field_tilt_pct', 'pass_tempo',
        'long_ball_pct', 'ball_in_play_minutes',
        'possessions_retained_after_5s', 'possessions_retained_after_5s_pct',
        'forward_passes_from_middle_third_pct',
        'buildups_from_goalkicks_pct',
    ],
    'Progression & Penetration': [
        'box_touches', 'box_entries', 'num_box_entries',
        'num_possessions_final_third', 'possessions_to_final_third_pct',
        'final_third_to_box_pct', 'box_to_shot_pct',
        'crosses_per_final_third_possession',
        'dribbles_per_final_third_possession',
        'box_entries_from_carries_pct', 'box_entries_from_crosses_pct',
        'possessions_to_box_within_10s_after_recovery',
        'possessions_to_final_third_within_10s_after_recovery',
    ],
    'Defensive': [
        'defensive_duels_won_pct', 'defensive_action_height_m',
        'recovery_line_height_m', 'defensive_intensity', 'ppda',
        'recoveries', 'num_recoveries_att_half', 'final_third_recoveries_pct',
        'fouls_commited', 'fouls_in_attacking_half_pct',
        'red_cards', 'yellow_cards',
    ],
    'Pressing & Transitions': [
        'time_to_defensive_action_s',
        'time_to_defensive_action_after_loss_own_half_s',
        'time_to_defensive_action_after_loss_att_half_s',
        'time_to_recovery_s', 'recoveries_within_5s_pct',
        'final_third_entry_within_10s_after_recovery_own_half_pct',
        'box_entry_within_10_after_recovery_att_half_pct',
        'first_pass_forward_after_recovery_own_half_pct',
        'median_time_to_first_forward_pass_own_half_s',
    ],
    'Set Pieces': [
        'corners', 'num_throwins_final_third', 'offsides',
    ],
    'Results & Points': [
        'goal_difference', 'points', 'xpts',
        'win_probability_pct', 'xpts_points_diff', 'match_duration_minutes',
    ],
}

# Verify coverage
all_mapped = [c for cols in domains.values() for c in cols]
unmapped = [c for c in metric_cols if c not in all_mapped]

print('DOMAIN SUMMARY')
print('=' * 50)
for domain, cols in domains.items():
    present = [c for c in cols if c in ts.columns]
    print(f'{domain:35s} {len(present):2d} metrics')
print(f'{"TOTAL":35s} {len(all_mapped):2d} metrics')

if unmapped:
    print(f'\nUnmapped columns: {unmapped}')

---
## 4. Distribution of Key Metrics

Quick look at the scale and spread of representative metrics from each domain.

In [ ]:
# Summary stats for all metrics
summary = ts[metric_cols].describe().T
summary['null_pct'] = ts[metric_cols].isna().mean()
summary = summary[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max', 'null_pct']]
summary = summary.round(3)
print(summary.to_string())

In [ ]:
# Histograms for a representative sample
sample_metrics = [
    'goals', 'xg', 'shots', 'ball_possession_pct',
    'ppda', 'defensive_intensity', 'defensive_action_height_m',
    'box_touches', 'np_xg_per_shot', 'goal_difference',
    'time_to_defensive_action_s', 'points',
]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, col in zip(axes.flat, sample_metrics):
    vals = ts[col].dropna()
    vals = vals[np.isfinite(vals)]
    ax.hist(vals, bins=50, color='steelblue', edgecolor='white', alpha=0.8)
    ax.set_title(col, fontsize=9)
    ax.axvline(vals.mean(), color='red', ls='--', lw=1)
    ax.tick_params(labelsize=7)

plt.suptitle('Distribution of Key Team Stats (red = mean)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## 5. Overlap with Master Parquet

The master parquet has `from_team_stats_*` and `to_team_stats_*` columns. Let's confirm:
1. Which metrics from team_stats_season made it into the master?
2. Are there any extras in the master not in team_stats_season?

In [ ]:
# Extract team stat column names from master parquet
from_team_cols = [c for c in master.columns if c.startswith('from_team_stats_')]
to_team_cols = [c for c in master.columns if c.startswith('to_team_stats_')]

from_metrics = [c.replace('from_team_stats_', '') for c in from_team_cols]
to_metrics = [c.replace('to_team_stats_', '') for c in to_team_cols]

print(f'Master parquet FROM team metrics: {len(from_metrics)}')
print(f'Master parquet TO team metrics:   {len(to_metrics)}')
print(f'team_stats_season metrics:        {len(metric_cols)}')

# Overlap
in_both = set(from_metrics) & set(metric_cols)
in_ts_only = set(metric_cols) - set(from_metrics)
in_master_only = set(from_metrics) - set(metric_cols)

print(f'\nIn both:             {len(in_both)}')
print(f'In team_stats only:  {len(in_ts_only)}')
print(f'In master only:      {len(in_master_only)}')

if in_ts_only:
    print(f'\nMetrics in team_stats_season NOT in master:')
    for c in sorted(in_ts_only):
        print(f'  - {c}')

if in_master_only:
    print(f'\nMetrics in master NOT in team_stats_season:')
    for c in sorted(in_master_only):
        print(f'  - {c}')

---
## 6. Mapping to Twelve's Team Quality Formulas

Twelve computes 7 team qualities from z-scored team stats. Let's see what we can and can't compute.

The formulas reference these variables (from Twelve's code):
- **DEFENCE**: goals_conceded, xga, defensive_intensity, ppda, time_to_def_action_inside, time_to_def_action_outside
- **DEFENSIVE_TRANSITION**: counter_attacks_against, counter_attack_goals_conceded, counter_attack_xga, counter_attack_shots_against
- **ATTACKING_TRANSITION**: counter_attacks, counter_attack_goals, counter_attack_xg, counter_attack_shots
- **ATTACK**: goals_scored, xg, shots, ball_possession_pct, passes_to_final_third
- **PENETRATION**: deep_completions, crosses_into_box, dribbles, touches_in_box, progressive_runs
- **CHANCE_CREATION**: xg_per_shot, shots_on_target_pct, shot_assists, key_passes, smart_passes
- **OUTCOME**: goals_scored, xg, goals_conceded, xga (+ derived: goal_diff, xg_diff, xgd_per_game)

In [ ]:
# Map Twelve formula variables -> team_stats_season columns
twelve_mapping = {
    # DEFENCE
    'goals_conceded':              None,  # opponent metric - NOT in team_stats
    'xga':                         None,  # opponent metric
    'defensive_intensity':         'defensive_intensity',
    'ppda':                        'ppda',
    'time_to_def_action_inside':   'time_to_defensive_action_after_loss_own_half_s',
    'time_to_def_action_outside':  'time_to_defensive_action_after_loss_att_half_s',
    # DEFENSIVE_TRANSITION
    'counter_attacks_against':           None,
    'counter_attack_goals_conceded':     None,
    'counter_attack_xga':               None,
    'counter_attack_shots_against':      None,
    # ATTACKING_TRANSITION
    'counter_attacks':             None,
    'counter_attack_goals':        None,
    'counter_attack_xg':           None,
    'counter_attack_shots':        None,
    # ATTACK
    'goals_scored':                'goals',
    'xg':                          'xg',
    'shots':                       'shots',
    'ball_possession_pct':         'ball_possession_pct',
    'passes_to_final_third':       None,  # not in team_stats as raw count
    # PENETRATION
    'deep_completions':            None,
    'crosses_into_box':            None,
    'dribbles':                    None,
    'touches_in_box':              'box_touches',
    'progressive_runs':            None,
    # CHANCE_CREATION
    'xg_per_shot':                 'np_xg_per_shot',
    'shots_on_target_pct':         None,  # can derive from shots_on_target / shots
    'shot_assists':                None,
    'key_passes':                  None,
    'smart_passes':                None,
    # OUTCOME
    'goal_diff':                   'goal_difference',
    'xg_diff':                     None,  # could derive if we had xga
    'xgd_per_game':               None,
}

available = {k: v for k, v in twelve_mapping.items() if v is not None}
missing = {k: v for k, v in twelve_mapping.items() if v is None}

print(f'TWELVE FORMULA VARIABLES')
print(f'=' * 60)
print(f'Available in team_stats_season: {len(available)}/{len(twelve_mapping)}')
print(f'Missing:                        {len(missing)}/{len(twelve_mapping)}')

print(f'\n--- AVAILABLE ---')
for var, col in available.items():
    print(f'  {var:45s} -> {col}')

print(f'\n--- MISSING ---')
for var in missing:
    print(f'  {var}')

In [ ]:
# Impact: which qualities CAN we fully compute?
quality_vars = {
    'DEFENCE': ['goals_conceded', 'xga', 'defensive_intensity', 'ppda',
                'time_to_def_action_inside', 'time_to_def_action_outside'],
    'DEFENSIVE_TRANSITION': ['counter_attacks_against', 'counter_attack_goals_conceded',
                             'counter_attack_xga', 'counter_attack_shots_against'],
    'ATTACKING_TRANSITION': ['counter_attacks', 'counter_attack_goals',
                             'counter_attack_xg', 'counter_attack_shots'],
    'ATTACK': ['goals_scored', 'xg', 'shots', 'ball_possession_pct', 'passes_to_final_third'],
    'PENETRATION': ['deep_completions', 'crosses_into_box', 'dribbles',
                    'touches_in_box', 'progressive_runs'],
    'CHANCE_CREATION': ['xg_per_shot', 'shots_on_target_pct', 'shot_assists',
                        'key_passes', 'smart_passes'],
    'OUTCOME': ['goals_scored', 'xg', 'goals_conceded', 'xga', 'goal_diff',
                'xg_diff', 'xgd_per_game'],
}

print('QUALITY COMPLETENESS')
print('=' * 60)
for quality, vars_list in quality_vars.items():
    n_available = sum(1 for v in vars_list if twelve_mapping.get(v) is not None)
    n_total = len(vars_list)
    status = 'FULL' if n_available == n_total else f'PARTIAL ({n_available}/{n_total})' if n_available > 0 else 'NONE'
    avail_vars = [v for v in vars_list if twelve_mapping.get(v) is not None]
    miss_vars = [v for v in vars_list if twelve_mapping.get(v) is None]
    
    print(f'\n{quality} — {status}')
    if avail_vars:
        print(f'  Available: {avail_vars}')
    if miss_vars:
        print(f'  Missing:   {miss_vars}')

---
## 7. What IS Available: The Full 74 Metrics

Since Twelve's exact formulas need variables we don't have, let's look at what we DO have and whether we can build **our own team quality features** from the 74 available metrics.

In [ ]:
# Correlation matrix of all metrics (subsample for speed)
sample = ts[metric_cols].sample(min(5000, len(ts)), random_state=42)

corr = sample.corr()

fig, ax = plt.subplots(figsize=(18, 15))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(metric_cols)))
ax.set_yticks(range(len(metric_cols)))
ax.set_xticklabels(metric_cols, rotation=90, fontsize=6)
ax.set_yticklabels(metric_cols, fontsize=6)
plt.colorbar(im, ax=ax, fraction=0.046)
ax.set_title('Correlation Matrix: All 74 Team Stats Metrics')
plt.tight_layout()
plt.show()

In [ ]:
# Highly correlated pairs (|r| > 0.85)
high_corr = []
for i in range(len(metric_cols)):
    for j in range(i+1, len(metric_cols)):
        r = corr.iloc[i, j]
        if abs(r) > 0.85:
            high_corr.append((metric_cols[i], metric_cols[j], r))

high_corr.sort(key=lambda x: -abs(x[2]))
print(f'Highly correlated pairs (|r| > 0.85): {len(high_corr)}')
print(f'{"":-<80}')
for a, b, r in high_corr[:30]:
    print(f'  {r:+.3f}  {a:50s} <-> {b}')

---
## 8. League-Level Context: Z-Score Reference Populations

For computing within-league z-scores, we need enough teams per (competition, season). Let's check population sizes.

In [ ]:
pop_sizes = ts.groupby(['competition_id', 'season']).size().reset_index(name='n_teams')

print('Population sizes for z-scoring:')
print(pop_sizes['n_teams'].describe().to_string())

print(f'\nDistribution of population sizes:')
bins = [0, 5, 10, 15, 20, 25, 30, 100]
labels = ['1-5', '6-10', '11-15', '16-20', '21-25', '26-30', '30+']
pop_sizes['bin'] = pd.cut(pop_sizes['n_teams'], bins=bins, labels=labels)
print(pop_sizes['bin'].value_counts().sort_index().to_string())

print(f'\nPopulations with < 6 teams (z-scores unreliable): {(pop_sizes["n_teams"] < 6).sum()}')
print(f'Populations with >= 10 teams (z-scores solid):    {(pop_sizes["n_teams"] >= 10).sum()}')

In [ ]:
# Focus: seasons 2018-2025 (our transfer window)
ts_model = ts[ts['season'].between(2018, 2025)].copy()
pop_model = ts_model.groupby(['competition_id', 'season']).size().reset_index(name='n_teams')

print(f'=== SEASONS 2018-2025 (model window) ===')
print(f'Rows: {len(ts_model):,}')
print(f'(Comp, Season) groups: {len(pop_model)}')
print(f'\nPopulation sizes:')
print(pop_model['n_teams'].describe().to_string())
print(f'\nWith < 6 teams:  {(pop_model["n_teams"] < 6).sum()}')
print(f'With >= 10 teams: {(pop_model["n_teams"] >= 10).sum()}')

---
## 9. Match with Transfer Data: Coverage Check

Can we find team stats for every (team, competition, season) referenced in the master transfer parquet?

In [ ]:
# FROM side: unique (team_id, competition_id, season) in transfers
# Note: master uses from_competition/to_competition and from_season/to_season
from_keys = master[['from_team_id', 'from_competition', 'from_season']].drop_duplicates()
from_keys.columns = ['team_id', 'competition_id', 'season']

to_keys = master[['to_team_id', 'to_competition', 'to_season']].drop_duplicates()
to_keys.columns = ['team_id', 'competition_id', 'season']

all_keys = pd.concat([from_keys, to_keys]).drop_duplicates()
ts_keys = ts[['team_id', 'competition_id', 'season']].drop_duplicates()

# Merge to find matches
merged = all_keys.merge(ts_keys, on=['team_id', 'competition_id', 'season'],
                        how='left', indicator=True)

n_found = (merged['_merge'] == 'both').sum()
n_missing = (merged['_merge'] == 'left_only').sum()

print(f'Unique (team, comp, season) in transfers: {len(all_keys):,}')
print(f'Found in team_stats_season:               {n_found:,} ({n_found/len(all_keys):.1%})')
print(f'NOT found:                                {n_missing:,} ({n_missing/len(all_keys):.1%})')

if n_missing > 0:
    missing_rows = merged[merged['_merge'] == 'left_only']
    print(f'\nMissing by season:')
    print(missing_rows['season'].value_counts().sort_index().to_string())

In [ ]:
# Same check but at transfer row level (how many transfers can we enrich?)
# Check FROM side
from_check = master[['from_team_id', 'from_competition', 'from_season']].copy()
from_check.columns = ['team_id', 'competition_id', 'season']
from_merged = from_check.merge(ts_keys, on=['team_id', 'competition_id', 'season'],
                                how='left', indicator=True)
from_hit = (from_merged['_merge'] == 'both').sum()

# Check TO side
to_check = master[['to_team_id', 'to_competition', 'to_season']].copy()
to_check.columns = ['team_id', 'competition_id', 'season']
to_merged = to_check.merge(ts_keys, on=['team_id', 'competition_id', 'season'],
                            how='left', indicator=True)
to_hit = (to_merged['_merge'] == 'both').sum()

# Both sides covered
both_covered = ((from_merged['_merge'] == 'both') & (to_merged['_merge'] == 'both')).sum()

n = len(master)
print(f'Transfer rows: {n:,}')
print(f'FROM team found: {from_hit:,} ({from_hit/n:.1%})')
print(f'TO team found:   {to_hit:,} ({to_hit/n:.1%})')
print(f'BOTH found:      {both_covered:,} ({both_covered/n:.1%})')

---
## 10. Summary & Decisions Needed

### What we have
- **74 raw team metrics** per (team, competition, season) — zero nulls
- **36,228 rows** covering 375 competitions, 7,529 teams, seasons 2014-2026
- Full coverage of offensive/process stats: shooting, xG, possession, pressing, build-up, transitions

### What we're missing for Twelve's exact formulas
- **All opponent/against metrics**: goals_conceded, xGA, counter-attacks against
- **All counter-attack metrics** (even own): counter_attacks, counter_attack_goals, etc.
- **Some creative team metrics**: deep_completions, crosses_into_box, dribbles, progressive_runs, shot_assists, key_passes, smart_passes

### Options
1. **Use the 74 available metrics directly** as model features (from_team + to_team = 148 features, or deltas = 74)
2. **Build partial team qualities** using only available metrics (skip DEFENSIVE_TRANSITION and ATTACKING_TRANSITION entirely, partial DEFENCE, ATTACK, etc.)
3. **Design our own team quality composites** based on what we have, inspired by Twelve's approach but adapted to our data

In [ ]:
print('Notebook complete. Review the outputs above and decide on the team quality strategy.')